# Task 2 (BONUS): Transformer (BERT/RoBERTa) Sentiment Classifier
Fine-tuning a pre-trained transformer on the Twitter US Airline Sentiment dataset.

**Pehle ye karo:** menu me `Runtime > Change runtime type > T4 GPU` select karo, phir har cell ko upar se neeche **Shift + Enter** se chalao.

## Cell 1: Libraries

In [ ]:
!pip -q install transformers scikit-learn seaborn
import re, copy, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cpu":
    print("WARNING: GPU on nahi hai! Runtime > Change runtime type > T4 GPU select karo.")

## Cell 2: Settings

In [ ]:
# Model choice. Dono me se koi bhi use kar sakti ho:
#   "cardiffnlp/twitter-roberta-base-sentiment-latest"  -> RoBERTa jo tweets par pre-train hua (best result)
#   "bert-base-uncased"                                 -> classic BERT
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

MAX_LEN    = 64      # tweets chhoti hoti hain
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

LABELS   = ["negative", "neutral", "positive"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

## Cell 3: Tweets.csv upload karo
Jab file chooser aaye to apni `Tweets.csv` select karo.

In [ ]:
from google.colab import files
uploaded = files.upload()

## Cell 4: Data load + split (train / validation / test)

In [ ]:
df = pd.read_csv("Tweets.csv")[["text", "airline_sentiment"]]
df.columns = ["text", "label"]
df = df.dropna(subset=["text", "label"]).drop_duplicates(subset="text").reset_index(drop=True)


def light_clean(t):
    # Transformers ko stop-words/lemmatization ki zaroorat nahi hoti (wo poora sentence samajhte hain),
    # isliye sirf URLs aur @mentions normalise karte hain.
    t = re.sub(r"http\S+|www\S+", "http", t)
    t = re.sub(r"@\w+", "@user", t)
    return t.strip()


df["text"] = df["text"].astype(str).apply(light_clean)
df["y"] = df["label"].map(label2id)
print(df["label"].value_counts(), "\n")

train_val, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["y"])
train, val = train_test_split(train_val, test_size=0.1, random_state=SEED, stratify=train_val["y"])
print("Train:", len(train), " Validation:", len(val), " Test:", len(test))

## Cell 5: Tokenizer + DataLoaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TweetDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.labels[i])
        return item


collator = DataCollatorWithPadding(tokenizer)
train_loader = torch.utils.data.DataLoader(TweetDS(train["text"], train["y"]), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=collator)
val_loader   = torch.utils.data.DataLoader(TweetDS(val["text"], val["y"]), batch_size=64, collate_fn=collator)
test_loader  = torch.utils.data.DataLoader(TweetDS(test["text"], test["y"]), batch_size=64, collate_fn=collator)

## Cell 6: Model load + training setup

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
).to(device)

# Imbalanced data ke liye class weights (negative zyada hain, positive kam)
weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=train["y"].values)
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))
print("Class weights:", dict(zip(LABELS, weights.round(2))))

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def evaluate(loader):
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(**batch).logits
            all_preds += logits.argmax(dim=-1).cpu().tolist()
            all_true += labels.cpu().tolist()
    return np.array(all_true), np.array(all_preds)

## Cell 7: Training (GPU par kuch minute lagte hain)

In [ ]:
best_f1, best_state = 0.0, None
start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()

    y_true, y_pred = evaluate(val_loader)
    val_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"Epoch {epoch}/{EPOCHS} | train loss {running/len(train_loader):.4f} | "
          f"val Macro F1 {val_f1:.4f} | val acc {accuracy_score(y_true, y_pred):.4f} | "
          f"{(time.time()-start)/60:.1f} min")
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})

model.load_state_dict(best_state)
print(f"\nBest validation Macro F1: {best_f1:.4f}")

## Cell 8: Final test results + confusion matrix

In [ ]:
y_true, y_pred = evaluate(test_loader)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("=== FINAL TEST RESULTS (Transformer) ===")
print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))
print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 : {macro_f1:.4f}")
print("\nComparison:")
print("  v1  TF-IDF + Logistic Regression : Macro F1 0.7248")
print("  v2  TF-IDF (word+char) + SVM     : Macro F1 0.7696")
print(f"  v3  Transformer ({MODEL_NAME.split('/')[-1]}) : Macro F1 {macro_f1:.4f}")

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title("Confusion Matrix (counts)")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Greens", xticklabels=LABELS, yticklabels=LABELS, ax=axes[1])
axes[1].set_title("Confusion Matrix (row-normalised)")
for ax in axes:
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix_bert.png", dpi=150)
plt.show()

# Misclassified examples
test_out = test.copy()
test_out["actual"] = [id2label[i] for i in y_true]
test_out["predicted"] = [id2label[i] for i in y_pred]
test_out[test_out["actual"] != test_out["predicted"]].head(50)[["text", "actual", "predicted"]] \
    .to_csv("misclassified_examples_bert.csv", index=False)

## Cell 9: Apne sentences test karo

In [ ]:
def predict_sentiment(sentence):
    model.eval()
    enc = tokenizer(light_clean(sentence), return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits.float(), dim=-1)[0].cpu().numpy()
    return id2label[int(probs.argmax())], float(probs.max())


for s in ["I love this product, it works great!",
          "Worst service ever, totally disappointed.",
          "The package arrived on Tuesday.",
          "The flight was not good at all.",
          "Thanks, the crew was not rude, really helpful!"]:
    label, conf = predict_sentiment(s)
    print(f"{s}  ->  {label} ({conf:.0%})")

## Cell 10: Files download karo

In [ ]:
files.download("confusion_matrix_bert.png")
files.download("misclassified_examples_bert.csv")